In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.inspection import permutation_importance
from catboost import Pool, CatBoostClassifier

In [3]:
pd.set_option('display.max_columns',None)
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/encoded_plus_df.csv'

df = pd.read_csv(url)
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')
df['Best of'] = df['Best of'].astype(str)

random_state=42

In [4]:
weather_cols = ['temperature_2m_mean','apparent_temperature_mean', 'precipitation_sum', 'rain_sum','wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_gusts_10m_mean',
              'wind_speed_10m_mean', 'relative_humidity_2m_mean','relative_humidity_2m_max', 'relative_humidity_2m_min','soil_temperature_0_to_100cm_mean',
              'soil_moisture_0_to_100cm_mean','wind_gusts_10m_min', 'wind_speed_10m_min']
for col in weather_cols:
    df[col] = df[col]*df['is_outdoor']

In [10]:
"""
Reformatea el dataset de formato ancho (playerA/playerB, rankA/rankB, etc.)
a formato largo (Jugador/Oponente), duplicando cada partido en 2 filas.

Fila 1 de cada partido: Jugador = A, Oponente = B -> target = 1 si ganó A
Fila 2 de cada partido: Jugador = B, Oponente = A -> target = 1 si ganó B (0 si ganó A)

Cada partido conserva un match_id para poder hacer split agrupado después.
"""
PATH = url
COL_WINNER = "target"

def reformatear_a_largo(df: pd.DataFrame, col_winner: str = COL_WINNER) -> pd.DataFrame:
    df = df.copy()
    df["match_id"] = df.index  # id único por partido original

    # Columnas que terminan en "A" o "B" -> tienen versión jugador/oponente
    cols_a = [c for c in df.columns if c.endswith("A") and c != "match_id"]
    cols_b = [c for c in df.columns if c.endswith("B") and c != "match_id"]

    # Validamos que cada columna "A" tenga su par "B"
    pares = []
    for ca in cols_a:
        base = ca[:-1]  # nombre sin la "A" final, ej. "rank" de "rankA"
        cb = base + "B"
        if cb in cols_b:
            pares.append((ca, cb, base))
        else:
            print(f"Aviso: {ca} no tiene par {cb}, se ignora en el reformateo")

    # Columnas que NO dependen de A/B (Location, Series, Surface, Fecha, etc.)
    cols_comunes = [
        c for c in df.columns
        if c not in cols_a + cols_b + [col_winner, "match_id"]
    ]

    print("Pares A/B detectados:")
    for ca, cb, base in pares:
        print(f"  {ca:12s} / {cb:12s} -> jugador_{base} / oponente_{base}")
    print(f"\nColumnas comunes (no se duplican distinto): {cols_comunes}")

    # ---- Fila 1: Jugador = A, Oponente = B ----
    fila_a = df[cols_comunes + ["match_id"]].copy()
    for ca, cb, base in pares:
        fila_a[f"jugador_{base}"] = df[ca]
        fila_a[f"oponente_{base}"] = df[cb]
    fila_a["Jugador"] = df["playerA"]
    fila_a["Oponente"] = df["playerB"]
    fila_a["target"] = df[col_winner]  # 1 si ganó A (=Jugador en esta fila)

    # ---- Fila 2: Jugador = B, Oponente = A ----
    fila_b = df[cols_comunes + ["match_id"]].copy()
    for ca, cb, base in pares:
        fila_b[f"jugador_{base}"] = df[cb]
        fila_b[f"oponente_{base}"] = df[ca]
    fila_b["Jugador"] = df["playerB"]
    fila_b["Oponente"] = df["playerA"]
    fila_b["target"] = 1 - df[col_winner]  # invertido: 1 si ganó B

    # ---- Unimos y ordenamos para que las 2 filas de cada partido queden juntas ----
    df_largo = pd.concat([fila_a, fila_b], ignore_index=True)
    df_largo = df_largo.sort_values(["match_id"]).reset_index(drop=True)

    return df_largo


if __name__ == "__main__":
    df_largo = reformatear_a_largo(df)

    print(f"\nDataset original: {df.shape[0]} partidos")
    print(f"Dataset largo:    {df_largo.shape[0]} filas ({df_largo['match_id'].nunique()} partidos)")

Aviso: prob_consensus_A no tiene par prob_consensus_B, se ignora en el reformateo
Aviso: bookmaker_std_A no tiene par bookmaker_std_B, se ignora en el reformateo
Pares A/B detectados:
  playerA      / playerB      -> jugador_player / oponente_player
  rankA        / rankB        -> jugador_rank / oponente_rank
  PtsA         / PtsB         -> jugador_Pts / oponente_Pts
  B365A        / B365B        -> jugador_B365 / oponente_B365
  MaxA         / MaxB         -> jugador_Max / oponente_Max
  AvgA         / AvgB         -> jugador_Avg / oponente_Avg
  setsA        / setsB        -> jugador_sets / oponente_sets
  B365ProbA    / B365ProbB    -> jugador_B365Prob / oponente_B365Prob
  ProbAvgA     / ProbAvgB     -> jugador_ProbAvg / oponente_ProbAvg
  ProbMaxA     / ProbMaxB     -> jugador_ProbMax / oponente_ProbMax
  logit_oddsA  / logit_oddsB  -> jugador_logit_odds / oponente_logit_odds
  winrate_5_A  / winrate_5_B  -> jugador_winrate_5_ / oponente_winrate_5_
  winrate_10_A / winrate_10_B 

In [11]:
df_largo = df_largo.drop(columns=['A1', 'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5', 'B5','setsPartido', 'jugador_player', 'oponente_player',
                                  'series_level_oe', 'round_encoded', 'is_best_of_5', 'surface_fe','is_outdoor', 'jugador_sets', 'oponente_sets'])

In [12]:
"""
Split train/test para el dataset largo (Jugador/Oponente, 2 filas por partido).

Reglas que tiene que cumplir el split:
1. Temporal: train = partidos más viejos, test = partidos más nuevos
   (nunca aleatorio, porque es una serie temporal real).
2. Agrupado por match_id: las 2 filas de un mismo partido (Jugador/Oponente
   y su espejo) NUNCA pueden quedar separadas entre train y test, porque
   eso sería leakage (el modelo vería el resultado del partido en train
   y el mismo partido invertido en test).

Como ambas filas de un match_id comparten la misma Fecha, cortar por fecha
ya garantiza la regla 2 automáticamente -- pero igual lo chequeamos con un
assert para estar seguros de que no se rompió en algún paso anterior.
"""

# =========================================================
# CONFIG
# =========================================================
COL_FECHA = "Fecha"
COL_MATCH_ID = "match_id"
FECHA_CORTE_TEST = "2024-01-01"   # ajustar según el rango de fechas de tu dataset


def split_train_test(df: pd.DataFrame, col_fecha: str, col_match_id: str, fecha_corte: str):
    df = df.copy()
    df[col_fecha] = pd.to_datetime(df[col_fecha])

    train = df[df[col_fecha] < fecha_corte].reset_index(drop=True)
    test = df[df[col_fecha] >= fecha_corte].reset_index(drop=True)

    # ---- Chequeo de integridad: ningún match_id partido entre train y test ----
    match_ids_train = set(train[col_match_id])
    match_ids_test = set(test[col_match_id])
    overlap = match_ids_train & match_ids_test
    assert len(overlap) == 0, (
        f"Leakage detectado: {len(overlap)} match_id(s) aparecen en train y test. "
        "Revisá que ambas filas de cada partido tengan la misma fecha."
    )

    # ---- Chequeo de que cada match_id tiene exactamente 2 filas dentro de su set ----
    for nombre, sub in [("train", train), ("test", test)]:
        conteo = sub[col_match_id].value_counts()
        incompletos = conteo[conteo != 2]
        if len(incompletos) > 0:
            print(f"Aviso: {len(incompletos)} match_id(s) en {nombre} no tienen exactamente 2 filas")

    print(f"Fecha de corte: {fecha_corte}")
    print(f"Train: {len(train):>6} filas | {train[col_match_id].nunique():>5} partidos | "
          f"{train[col_fecha].min().date()} -> {train[col_fecha].max().date()}")
    print(f"Test:  {len(test):>6} filas | {test[col_match_id].nunique():>5} partidos | "
          f"{test[col_fecha].min().date()} -> {test[col_fecha].max().date()}")

    return train, test


if __name__ == "__main__":
    train, test = split_train_test(df_largo, COL_FECHA, COL_MATCH_ID, FECHA_CORTE_TEST)

Fecha de corte: 2024-01-01
Train:  25508 filas | 12754 partidos | 2015-01-05 -> 2023-11-05
Test:    6614 filas |  3307 partidos | 2024-01-14 -> 2025-11-02


In [36]:
# train.drop(columns=['match_id','fecha'], inplace=True)
# test.drop(columns=['match_id','fecha'], inplace=True)

# X_train = train.drop(columns=['target'])
# y_train = train['target']
# X_test = test.drop(columns=['target'])
# y_test = test['target']

In [13]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

# =========================================================
# 3. ENTRENAMIENTO CATBOOST
# =========================================================
def entrenar_catboost(train, test, features, cat_features, target_col):
    train_pool = Pool(
        data=train[features],
        label=train[target_col],
        cat_features=cat_features,
    )
    test_pool = Pool(
        data=test[features],
        label=test[target_col],
        cat_features=cat_features,
    )

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="Logloss",
        early_stopping_rounds=50,
        random_seed=42,
        verbose=100,
        nan_mode="Min",  # NaN nativos: los manda al extremo, el modelo aprende
                          # a tratarlos como "sin historial" en vez de borrarlos
    )

    model.fit(train_pool, eval_set=test_pool)
    return model, test_pool


def evaluar(model, test_pool, test_df, target_col, nombre_modelo):
    preds_proba = model.predict_proba(test_pool)[:, 1]
    preds = (preds_proba >= 0.5).astype(int)

    acc = accuracy_score(test_df[target_col], preds)
    ll = log_loss(test_df[target_col], preds_proba)
    brier = brier_score_loss(test_df[target_col], preds_proba)

    print(f"\n=== {nombre_modelo} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Log Loss:  {ll:.4f}")
    print(f"Brier:     {brier:.4f}")
    return {"modelo": nombre_modelo, "accuracy": acc, "log_loss": ll, "brier": brier}


train.drop(columns=['match_id','Fecha'], inplace=True)
test.drop(columns=['match_id','Fecha'], inplace=True)
resultados = []
COL_TARGET='target'
CAT_FEATURES = train.select_dtypes('object').columns.tolist()
CAT_FEATURES.remove('Jugador')
CAT_FEATURES.remove('Oponente')
COL_JUGADOR='Jugador'
COL_RIVAL='Oponente'
NUMERIC_FEATURES = train.select_dtypes('number').columns.tolist()
NUMERIC_FEATURES.remove('target')

# ---- Modelo SIN nombre de jugador (baseline con tus features históricas) ----
features_sin_nombre = NUMERIC_FEATURES + CAT_FEATURES
model_sin, pool_sin = entrenar_catboost(
    train, test, features_sin_nombre, CAT_FEATURES, COL_TARGET
)
resultados.append(
    evaluar(model_sin, pool_sin, test, COL_TARGET, "Sin nombre jugador")
)

# ---- Modelo CON nombre de jugador/rival ----
features_con_nombre = NUMERIC_FEATURES + CAT_FEATURES + [COL_JUGADOR, COL_RIVAL]
cat_features_con = CAT_FEATURES + [COL_JUGADOR, COL_RIVAL]
model_con, pool_con = entrenar_catboost(
    train, test, features_con_nombre, cat_features_con, COL_TARGET
)
resultados.append(
    evaluar(model_con, pool_con, test, COL_TARGET, "Con nombre jugador")
)

# ---- Comparación final ----
print("\n" + "=" * 50)
print("COMPARACIÓN FINAL")
print("=" * 50)
print(pd.DataFrame(resultados).to_string(index=False))

# ---- Feature importance del modelo con nombre ----
print("\nFeature importance (modelo con nombre):")
importances = model_con.get_feature_importance(pool_con, prettified=True)
print(importances.head(15).to_string(index=False))

0:	learn: 0.6844085	test: 0.6853290	best: 0.6853290 (0)	total: 87.4ms	remaining: 1m 27s
100:	learn: 0.5490534	test: 0.5735722	best: 0.5735722 (100)	total: 7.24s	remaining: 1m 4s
200:	learn: 0.5418536	test: 0.5729331	best: 0.5727061 (164)	total: 15.7s	remaining: 1m 2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5727060576
bestIteration = 164

Shrink model to first 165 iterations.

=== Sin nombre jugador ===
Accuracy:  0.6969
Log Loss:  0.5727
Brier:     0.1957
0:	learn: 0.6845242	test: 0.6856986	best: 0.6856986 (0)	total: 96.5ms	remaining: 1m 36s
100:	learn: 0.5365631	test: 0.5780197	best: 0.5749596 (83)	total: 9.21s	remaining: 1m 22s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5749596296
bestIteration = 83

Shrink model to first 84 iterations.

=== Con nombre jugador ===
Accuracy:  0.6976
Log Loss:  0.5750
Brier:     0.1964

COMPARACIÓN FINAL
            modelo  accuracy  log_loss    brier
Sin nombre jugador  0.696855  0.572706 0.195696
C